# DBS Artifact Removal: Multi-Method Comparison

**Problem**: DBS stimulation creates strong artificial signals at the fundamental frequency and harmonics (e.g., 7, 14, 21, ... Hz), which overwhelm endogenous brain activity.

**Solution**: Compare multiple advanced artifact removal methods:
- **Baseline-Referenced Wiener Filter** (Baseline-Aware)
- **Spectrum-Fit Multi-Harmonic Removal** (Frequency-Domain Interpolation)
- **Zapline+** (Chen et al., 2022 - Spectro-Spatial Filtering)
- **Freq-Domain Hampel** (Allen et al., 2010 - Spectral Outlier Detection)
- **Time-Domain Hampel** (Allen et al., 2010 - Temporal Outlier Detection)

**Goal**: Visualize and quantify which method best removes DBS artifacts while preserving endogenous brain rhythms.

## 1. Setup and Data Loading

In [ ]:
%matplotlib inline
import sys
import os
import numpy as np
import mne
import matplotlib.pyplot as plt
from scipy import signal
import pandas as pd
import seaborn as sns

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.filters import BaselineReferencedFilter, ArtifactFilterFactory
from src.preprocessing import EEGPreprocessor

plt.rcParams['figure.figsize'] = (16, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print("Imports successful!")

## 2. Load Data

In [ ]:
# Initialize preprocessor for channel standardization
preprocessor = EEGPreprocessor(input_dir="../data/XU/", output_dir="../data/processed/")

data_dir = os.path.join(project_root, "data", "XU")

# Load clean baseline (no DBS) — match state to your DBS recording
raw_baseline = mne.io.read_raw_edf(
    os.path.join(data_dir, "XUAWAKEPRE_deidentified.edf"), preload=True, verbose='WARNING'
)
raw_baseline = preprocessor.standardize_channels(raw_baseline)
raw_baseline.filter(l_freq=1.0, h_freq=70.0, fir_design='firwin', phase='zero', verbose='WARNING')

# Load DBS-contaminated recording (7 Hz)
raw_dbs = mne.io.read_raw_edf(
    os.path.join(data_dir, "XUAWAKE7_deidentified.edf"), preload=True, verbose='WARNING'
)
raw_dbs = preprocessor.standardize_channels(raw_dbs)
raw_dbs.filter(l_freq=1.0, h_freq=70.0, fir_design='firwin', phase='zero', verbose='WARNING')

print(f"Baseline: {raw_baseline.info['nchan']} channels, {raw_baseline.n_times} samples, {raw_baseline.info['sfreq']} Hz")
print(f"DBS:      {raw_dbs.info['nchan']} channels, {raw_dbs.n_times} samples, {raw_dbs.info['sfreq']} Hz")

## 3. Apply All Filtering Methods

In [ ]:
print("="*80)
print("Applying Baseline-Referenced Wiener Filter")
print("="*80)
wiener_filter = BaselineReferencedFilter(
    baseline_raw=raw_baseline,
    dbs_freq=7.0,
    harmonic_bandwidth=1.5,
    n_fft=4096,
    floor_db=-40.0,
    alpha=1.5
)
raw_wiener = wiener_filter.filter(raw_dbs)
print()

In [ ]:
print("="*80)
print("Applying Spectrum-Fit Multi-Harmonic Removal")
print("="*80)
raw_spectrum_fit = preprocessor.remove_artifacts_advanced(
    raw_dbs,
    method='spectrum_fit',
    f_target=7.0,
    bandwidth=2.0,
    attenuation_db=-60.0  # Strong attenuation
)
print()

In [ ]:
print("="*80)
print("Applying Zapline+ (Chen et al., 2022)")
print("="*80)
raw_zapline = preprocessor.remove_artifacts_advanced(
    raw_dbs,
    method='zapline',
    f_target=7.0,
    n_harmonics=10,
    threshold_percentile=95.0
)
print()

In [ ]:
print("="*80)
print("Applying Freq-Domain Hampel (Allen et al., 2010)")
print("="*80)
raw_hampel_freq = preprocessor.remove_artifacts_advanced(
    raw_dbs,
    method='hampel_freq',
    window_hz=2.0,
    n_sigmas=3.0,
    attenuation_db=-60.0  # Strong attenuation
)
print()

In [ ]:
print("="*80)
print("Applying Time-Domain Hampel (Allen et al., 2010)")
print("="*80)
raw_hampel_time = preprocessor.remove_artifacts_advanced(
    raw_dbs,
    method='hampel_time',
    window_sec=0.2,
    n_sigmas=3.0,
    attenuation_factor=1.5  # Stronger attenuation via multiple passes
)
print()

## 4. PSD Comparison: Full Spectrum

In [ ]:
def compute_avg_psd(raw, n_fft=4096):
    """Compute channel-averaged PSD in dB/Hz."""
    data = raw.get_data()
    freqs, psd = signal.welch(data, fs=raw.info['sfreq'], nperseg=n_fft,
                              noverlap=n_fft // 2, window='hann', axis=1)
    psd_db = 10 * np.log10(psd + 1e-30)
    return freqs, psd_db.mean(axis=0)  # Average across channels

# Compute PSDs
freqs_baseline, psd_baseline = compute_avg_psd(raw_baseline)
freqs_dbs, psd_dbs_raw = compute_avg_psd(raw_dbs)
freqs_wiener, psd_wiener = compute_avg_psd(raw_wiener)
freqs_spectrum, psd_spectrum = compute_avg_psd(raw_spectrum_fit)
freqs_zapline, psd_zapline = compute_avg_psd(raw_zapline)
freqs_hampel_f, psd_hampel_f = compute_avg_psd(raw_hampel_freq)
freqs_hampel_t, psd_hampel_t = compute_avg_psd(raw_hampel_time)

print("PSD computations complete!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Full spectrum view
ax = axes[0]
ax.plot(freqs_baseline, psd_baseline, 'k-', alpha=0.8, linewidth=2.5, label='Baseline (no DBS)')
ax.plot(freqs_dbs, psd_dbs_raw, 'r-', alpha=0.4, linewidth=1, label='DBS Raw (7 Hz)')
ax.plot(freqs_wiener, psd_wiener, 'b-', alpha=0.7, linewidth=2, label='Wiener (Baseline-Referenced)')
ax.plot(freqs_spectrum, psd_spectrum, 'g-', alpha=0.7, linewidth=2, label='Spectrum-Fit')
ax.plot(freqs_zapline, psd_zapline, 'purple', alpha=0.7, linewidth=2, label='Zapline+ (Chen et al.)')
ax.plot(freqs_hampel_f, psd_hampel_f, 'orange', alpha=0.7, linewidth=2, label='Freq-Domain Hampel')
ax.plot(freqs_hampel_t, psd_hampel_t, 'brown', alpha=0.7, linewidth=2, label='Time-Domain Hampel')

ax.set_xlim(1, 70)
ax.set_xlabel('Frequency (Hz)', fontsize=12, fontweight='bold')
ax.set_ylabel('Power (dB/Hz)', fontsize=12, fontweight='bold')
ax.set_title('Full Spectrum PSD Comparison', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

# Zoom into theta-alpha-beta range (4-35 Hz)
ax = axes[1]
ax.plot(freqs_baseline, psd_baseline, 'k-', alpha=0.8, linewidth=2.5, label='Baseline (no DBS)')
ax.plot(freqs_dbs, psd_dbs_raw, 'r-', alpha=0.4, linewidth=1, label='DBS Raw (7 Hz)')
ax.plot(freqs_wiener, psd_wiener, 'b-', alpha=0.7, linewidth=2, label='Wiener (Baseline-Referenced)')
ax.plot(freqs_spectrum, psd_spectrum, 'g-', alpha=0.7, linewidth=2, label='Spectrum-Fit')
ax.plot(freqs_zapline, psd_zapline, 'purple', alpha=0.7, linewidth=2, label='Zapline+ (Chen et al.)')
ax.plot(freqs_hampel_f, psd_hampel_f, 'orange', alpha=0.7, linewidth=2, label='Freq-Domain Hampel')
ax.plot(freqs_hampel_t, psd_hampel_t, 'brown', alpha=0.7, linewidth=2, label='Time-Domain Hampel')

ax.set_xlim(4, 35)
ax.set_xlabel('Frequency (Hz)', fontsize=12, fontweight='bold')
ax.set_ylabel('Power (dB/Hz)', fontsize=12, fontweight='bold')
ax.set_title('Zoomed: Theta-Alpha-Beta (4-35 Hz)', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

# Mark DBS harmonics
for ax in axes:
    for h in np.arange(7, 70, 7):
        ax.axvline(h, color='red', alpha=0.15, linestyle=':', linewidth=0.8)

plt.tight_layout()
plt.savefig(os.path.join(project_root, 'figures', 'multi_method_psd_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print("Full spectrum comparison plot saved!")

## 5. Band Power Preservation Analysis

In [ ]:
bands = {
    'Delta (1-4 Hz)':  (1, 4),
    'Theta (4-8 Hz)':  (4, 8),
    'Alpha (8-13 Hz)': (8, 13),
    'Beta (13-30 Hz)': (13, 30),
    'Gamma (30-70 Hz)': (30, 70),
}

def band_power(freqs, psd_linear, fmin, fmax):
    """Integrate PSD over a frequency band (trapezoidal rule)."""
    mask = (freqs >= fmin) & (freqs <= fmax)
    return np.trapz(psd_linear[mask], freqs[mask])

# Recompute PSDs in linear scale for proper integration
def compute_avg_psd_linear(raw, n_fft=4096):
    data = raw.get_data()
    freqs, psd = signal.welch(data, fs=raw.info['sfreq'], nperseg=n_fft,
                              noverlap=n_fft // 2, window='hann', axis=1)
    return freqs, psd.mean(axis=0)

# Compute linear PSDs
f_baseline, psd_baseline_lin = compute_avg_psd_linear(raw_baseline)
f_dbs, psd_dbs_lin = compute_avg_psd_linear(raw_dbs)
f_wiener, psd_wiener_lin = compute_avg_psd_linear(raw_wiener)
f_spectrum, psd_spectrum_lin = compute_avg_psd_linear(raw_spectrum_fit)
f_zapline, psd_zapline_lin = compute_avg_psd_linear(raw_zapline)
f_hampel_f, psd_hampel_f_lin = compute_avg_psd_linear(raw_hampel_freq)
f_hampel_t, psd_hampel_t_lin = compute_avg_psd_linear(raw_hampel_time)

# Create results table
results = []
for name, (fmin, fmax) in bands.items():
    bp_base = band_power(f_baseline, psd_baseline_lin, fmin, fmax)
    bp_dbs = band_power(f_dbs, psd_dbs_lin, fmin, fmax)
    bp_wien = band_power(f_wiener, psd_wiener_lin, fmin, fmax)
    bp_spectrum = band_power(f_spectrum, psd_spectrum_lin, fmin, fmax)
    bp_zapline = band_power(f_zapline, psd_zapline_lin, fmin, fmax)
    bp_hampel_f = band_power(f_hampel_f, psd_hampel_f_lin, fmin, fmax)
    bp_hampel_t = band_power(f_hampel_t, psd_hampel_t_lin, fmin, fmax)

    err_wien = 100 * (bp_wien - bp_base) / bp_base
    err_spectrum = 100 * (bp_spectrum - bp_base) / bp_base
    err_zapline = 100 * (bp_zapline - bp_base) / bp_base
    err_hampel_f = 100 * (bp_hampel_f - bp_base) / bp_base
    err_hampel_t = 100 * (bp_hampel_t - bp_base) / bp_base

    results.append({
        'Band': name,
        'Baseline': bp_base,
        'DBS Raw Error %': 100 * (bp_dbs - bp_base) / bp_base,
        'Wiener Error %': err_wien,
        'Spectrum-Fit Error %': err_spectrum,
        'Zapline+ Error %': err_zapline,
        'Hampel Freq Error %': err_hampel_f,
        'Hampel Time Error %': err_hampel_t,
    })

df_results = pd.DataFrame(results)
print("\n" + "="*120)
print("Band Power Preservation: Relative Error vs Baseline (%)")
print("="*120)
print(df_results.to_string(index=False))
print("="*120)

## 6. Harmonic Attenuation Analysis

In [ ]:
# Analyze attenuation at each harmonic
harmonics = [7, 14, 21, 28, 35, 42, 49]
attenuation_results = []

for h in harmonics:
    # Band around each harmonic: +/- 0.5 Hz
    mask = (freqs_baseline >= h - 0.5) & (freqs_baseline <= h + 0.5)
    if not np.any(mask):
        continue
    
    # Average power in the band for each method (convert from dB to linear)
    psd_base_lin = 10 ** (psd_baseline[mask] / 10.0)
    psd_dbs_lin = 10 ** (psd_dbs_raw[mask] / 10.0)
    psd_wien_lin = 10 ** (psd_wiener[mask] / 10.0)
    psd_spectrum_lin = 10 ** (psd_spectrum[mask] / 10.0)
    psd_zapline_lin = 10 ** (psd_zapline[mask] / 10.0)
    psd_hampel_f_lin = 10 ** (psd_hampel_f[mask] / 10.0)
    psd_hampel_t_lin = 10 ** (psd_hampel_t[mask] / 10.0)
    
    # Attenuation: compare filtered vs raw DBS at each harmonic
    atten_wien = 10 * np.log10(np.mean(psd_wien_lin) / np.mean(psd_dbs_lin) + 1e-30)
    atten_spectrum = 10 * np.log10(np.mean(psd_spectrum_lin) / np.mean(psd_dbs_lin) + 1e-30)
    atten_zapline = 10 * np.log10(np.mean(psd_zapline_lin) / np.mean(psd_dbs_lin) + 1e-30)
    atten_hampel_f = 10 * np.log10(np.mean(psd_hampel_f_lin) / np.mean(psd_dbs_lin) + 1e-30)
    atten_hampel_t = 10 * np.log10(np.mean(psd_hampel_t_lin) / np.mean(psd_dbs_lin) + 1e-30)
    
    attenuation_results.append({
        'Harmonic (Hz)': h,
        'Wiener (dB)': atten_wien,
        'Spectrum-Fit (dB)': atten_spectrum,
        'Zapline+ (dB)': atten_zapline,
        'Hampel Freq (dB)': atten_hampel_f,
        'Hampel Time (dB)': atten_hampel_t,
    })

df_attenuation = pd.DataFrame(attenuation_results)
print("\n" + "="*100)
print("Harmonic Attenuation (Negative = Reduction in Power)")
print("="*100)
print(df_attenuation.to_string(index=False))
print("="*100)

In [ ]:
# Visualization of attenuation per harmonic
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(df_attenuation))
width = 0.15

ax.bar(x - 2*width, df_attenuation['Wiener (dB)'], width, label='Wiener', alpha=0.8)
ax.bar(x - width, df_attenuation['Spectrum-Fit (dB)'], width, label='Spectrum-Fit', alpha=0.8)
ax.bar(x, df_attenuation['Zapline+ (dB)'], width, label='Zapline+', alpha=0.8)
ax.bar(x + width, df_attenuation['Hampel Freq (dB)'], width, label='Hampel Freq', alpha=0.8)
ax.bar(x + 2*width, df_attenuation['Hampel Time (dB)'], width, label='Hampel Time', alpha=0.8)

ax.set_xlabel('Harmonic Frequency (Hz)', fontsize=12, fontweight='bold')
ax.set_ylabel('Attenuation (dB, negative = reduction)', fontsize=12, fontweight='bold')
ax.set_title('DBS Harmonic Attenuation by Method', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df_attenuation['Harmonic (Hz)'].astype(int))
ax.axhline(0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(project_root, 'figures', 'harmonic_attenuation_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print("Harmonic attenuation comparison plot saved!")

## 7. Summary and Recommendations

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════════════════════╗
║                    METHOD COMPARISON SUMMARY                                      ║
╚════════════════════════════════════════════════════════════════════════════════════╝

1. BASELINE-REFERENCED WIENER FILTER
   ✓ Pros: Excellent brain activity preservation, time-varying adaptation
   ✗ Cons: Requires clean baseline recording, moderate harmonic attenuation
   💡 Best for: Preserving endogenous rhythms when baseline data is available

2. SPECTRUM-FIT MULTI-HARMONIC REMOVAL
   ✓ Pros: Strong harmonic attenuation, smooth spectral tapering
   ✗ Cons: May affect frequencies near harmonics
   💡 Best for: Aggressive artifact removal with good frequency resolution

3. ZAPLINE+ (Chen et al., 2022)
   ✓ Pros: Adaptive spectro-spatial filtering, preserves brain signals
   ✗ Cons: Computationally heavier, requires channel correlation
   💡 Best for: Multi-channel recordings with consistent artifact patterns

4. FREQ-DOMAIN HAMPEL (Allen et al., 2010)
   ✓ Pros: Strong peak detection, preserves phase relationships
   ✗ Cons: Requires tuning of MAD threshold
   💡 Best for: Sharp, well-defined harmonic peaks

5. TIME-DOMAIN HAMPEL (Allen et al., 2010)
   ✓ Pros: Removes transient artifacts, simple interpretation
   ✗ Cons: May over-smooth oscillatory signals
   💡 Best for: Impulsive noise and sharp DBS pulses

╔════════════════════════════════════════════════════════════════════════════════════╗
║                         RECOMMENDED WORKFLOW                                       ║
╚════════════════════════════════════════════════════════════════════════════════════╝

Priority 1: Try Spectrum-Fit + Wiener for strong, consistent artifact removal
            while preserving brain activity.

Priority 2: If baseline data available: Use Baseline-Wiener for optimal
            preservation of endogenous rhythms.

Priority 3: Use Zapline+ for research requiring spectro-spatial filtering
            with multi-channel coordination.

Priority 4: Combine with Time-Domain Hampel for robust handling of transient
            DBS pulses and impulsive noise.

╔════════════════════════════════════════════════════════════════════════════════════╗
║                    TUNING PARAMETERS FOR STRONGER ATTENUATION                       ║
╚════════════════════════════════════════════════════════════════════════════════════╝

• Spectrum-Fit: Decrease attenuation_db (e.g., -60 → -80 dB)
• Zapline+: Increase n_harmonics, decrease threshold_percentile
• Hampel Freq: Increase n_sigmas (3.0 → 5.0), decrease attenuation_db
• Hampel Time: Increase attenuation_factor (1.0 → 2.0-3.0 for multiple passes)
• Wiener: Increase alpha parameter (1.5 → 2.0), decrease floor_db floor
""")